# 5. Measured boiling points: pre-registered Tests A and B, Table 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Singati2/LOYOLA-PAPER/blob/review/corrections-impl-2026-10-19/colab/05_boiling_points.ipynb)

Runs these scripts in this order (about 226 minutes on the lab machine):

1. `external/bp/coverage_bias.py` (lab: 1 min)
2. `external/bp/wiener_baseline.py` (lab: 1 min)
3. `external/bp/bp_tests.py` (lab: 78 min)
4. `external/bp/make_bp_table.py` (lab: 1 min)
5. `external/bp/selection_resampling.py` (lab: 145 min)

Each script rewrites its committed output files; notebook 11 compares them with the commit. If the session disconnects, open this notebook again and run all cells: finished scripts are skipped and the run continues with the next one. Keep the browser tab open while it runs.

In [ ]:
# Setup (the same cell in every notebook of colab/; about 2 minutes the first time in a session)
import base64, hashlib, json, os, shutil, subprocess, sys


def git_net(args, token=None, cwd=None):
    """Run a git command that may need the network; on failure raise with the token redacted."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    secrets = []
    if token:
        auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
        secrets = [token, auth]
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0="Authorization: Basic " + auth)
    try:
        r = subprocess.run(["git", *args], cwd=cwd, env=env, capture_output=True, text=True)
        msg = (r.stdout + r.stderr).strip()
        for s in secrets:
            msg = msg.replace(s, "***")
    finally:
        env = secrets = auth = None
    if r.returncode != 0:
        raise RuntimeError("git " + args[0] + " failed: " + msg[-800:])


def at_pin(work, pin, fetch_args, token=None):
    """Make the checkout at work be exactly the commit pin, with no local changes, or stop."""
    git = lambda *a: subprocess.run(["git", "-C", work, *a], capture_output=True, text=True)
    if git("status", "--porcelain", "--untracked-files=no").stdout.strip():
        raise RuntimeError(work + " has local changes to tracked files; delete that folder and run this cell again")
    if git("rev-parse", "HEAD").stdout.strip() != pin:
        if git("cat-file", "-e", pin + "^{commit}").returncode != 0:
            git_net(["fetch", "--quiet", *fetch_args], token, cwd=work)
        r = git("checkout", "--quiet", "--detach", pin)
        if r.returncode != 0:
            raise RuntimeError("cannot check out the pinned commit " + pin + ": " + r.stderr.strip())
    head = git("rev-parse", "HEAD").stdout.strip()
    if head != pin:
        raise RuntimeError("checkout is at " + head + ", not the pinned commit " + pin)
    return head

REVISION = "review/corrections-impl-2026-10-19"   # the branch or tag that is cloned; it must contain PIN
PIN = os.environ.get("LOYOLA_PIN", "549203c8311741f53c67bdc8ce466d22069daae0")   # the exact commit every notebook of a run executes
REPO_URL = "https://github.com/Singati2/LOYOLA-PAPER.git"

IN_COLAB = "google.colab" in sys.modules
token = None
try:
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")             # progress and outputs are kept in MyDrive/LOYOLA_colab_run
        WORK = "/content/LOYOLA-PAPER"
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")  # Colab secret (key icon on the left), needed while the repository is private
        except Exception:
            pass
        bundle = "/content/drive/MyDrive/LOYOLA-PAPER.bundle"
        source = REPO_URL if token or not os.path.exists(bundle) else bundle
        if not os.path.exists(WORK):
            git_net(["clone", "--quiet", "--branch", REVISION, source, WORK], token)
        at_pin(WORK, PIN, [source, REVISION], token)   # an existing checkout is checked, and moved to PIN if clean
        os.chdir(WORK)
        ENV = "/content/loyola-env"
    else:                                         # run locally from inside the repository
        while not os.path.exists("verify.py") and os.getcwd() != "/":
            os.chdir("..")
        head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
        if head != PIN or subprocess.run(["git", "status", "--porcelain", "--untracked-files=no"],
                                         capture_output=True, text=True).stdout.strip():
            raise RuntimeError("this checkout is " + head[:9] + (" with local changes" if head == PIN else "")
                               + ", not the pinned commit " + PIN[:9] + "; check out the pinned commit, or set LOYOLA_PIN")
        ENV = os.path.join(os.getcwd(), ".venv-colab")
finally:
    token = None                                  # the token is not kept in the notebook's variables

# Python 3.13 and exactly the locked package versions that produced the committed outputs. An environment
# with another Python is rebuilt; uv pip sync installs what is missing and removes what is not locked, so a
# partly installed or modified environment is repaired. Its fingerprint is printed and checked by pipeline.py.
PY = os.path.join(ENV, "bin", "python")
if os.path.exists(PY) and not subprocess.run([PY, "-c", "import sys; sys.exit(sys.version_info[:2] != (3, 13))"]).returncode == 0:
    shutil.rmtree(ENV)
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
UV = ["uv"] if shutil.which("uv") else [sys.executable, "-m", "uv"]
if not os.path.exists(PY):
    subprocess.run(UV + ["venv", "--quiet", "--python", "3.13", ENV], check=True)
subprocess.run(UV + ["pip", "sync", "--quiet", "--python", PY, "requirements-lock.txt", "requirements-extras-lock.txt"], check=True)
locks = hashlib.sha256(b"".join(open(f, "rb").read() for f in ("requirements-lock.txt", "requirements-extras-lock.txt"))).hexdigest()
print(subprocess.run([PY, "-c", "import sys, numpy, scipy; print('python', sys.version.split()[0], '| numpy', "
                      "numpy.__version__, '| scipy', scipy.__version__)"], capture_output=True, text=True).stdout.strip())
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(), "| lock files", locks[:12])
STATE = os.environ.get("LOYOLA_STATE") or ("/content/drive/MyDrive/LOYOLA_colab_run" if IN_COLAB
                                           else os.path.join(os.getcwd(), "colab_run"))   # as in pipeline.py


def pipeline(*args):
    """Run colab/pipeline.py with the pinned Python, show its output as it comes, stop on failure."""
    p = subprocess.Popen([PY, "-u", "colab/pipeline.py", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError("pipeline.py " + " ".join(args) + " reported a problem (see the output above)")


In [ ]:
pipeline("--stage", "5")